In [ ]:
import lightkurve
import matplotlib.pyplot as plot
import numpy
%matplotlib inline
plot.style.use('seaborn-v0_8')

In [ ]:
star_id = [
    4614514,
    4782712,
    7203984,
    7551387,
    25133286
][-1] # -1 for last too

star_id = f'TIC {star_id}'
star_id

In [ ]:
custom_id = None
if custom_id is not None and custom_id != '':
    star_id = custom_id

star_id

In [ ]:
search_result = lightkurve.search_targetpixelfile(star_id, mission = 'TESS')
search_result

In [ ]:
pixel_files = search_result[:10].download_all()
pixel_files

In [ ]:
collection = lightkurve.LightCurveCollection(None)
for pixel_file in pixel_files[:]:
    pixel_file = pixel_file[numpy.isfinite(pixel_file.flux).all(axis = (1, 2))]
    pixel_file = pixel_file[numpy.isfinite(pixel_file.flux_err).all(axis = (1, 2))]

    lc = pixel_file.to_lightcurve(aperture_mask = pixel_file.pipeline_mask)

    lc.errorbar()
    collection.append(lc)

plot.show()

In [ ]:
lc = collection[:].stitch().normalize()
lc = lc#.flatten(window_length = 501, break_tolerance = 10, niters = 1, sigma = 10)#.remove_outliers(sigma = 3)
# lc = lc.remove_outliers(sigma = 5)
lc.errorbar()
# plot.gca().set_xlim(1870, 1875)
plot.show()

In [ ]:
periods = numpy.logspace(-1.5, 1, 10 ** 3, base = 10.0)
periodogram = lc.to_periodogram(method = 'ls', period = periods)
periodogram.plot()
plot.show()

In [ ]:
period = periodogram.period_at_max_power
# transit_time = periodogram.transit_time_at_max_power
# transit_duration = periodogram.duration_at_max_power
period#, transit_time, transit_duration

In [ ]:
# transit_time += period / 2 # 2nd eclipse
# period *= 2 # 2nd eclipse
# period /= 2 # wrong period
folded_lc = lc.fold(period = period, epoch_time = 0)
binned_lc = folded_lc.bin(time_bin_size = 0.1)
ax = folded_lc.scatter()
binned_lc.plot(ax = ax, c = 'r')
d = 1
# ax.set_xlim(-d, d)
# ax.set_xlim(-3.4, -2.6)
# ax.set_ylim(0.998, 1.002)
plot.show()
transit_depth = 1 - numpy.nanmin(binned_lc.flux.value)
transit_depth